# Prueba del módulo `limpieza.py`
Este notebook prueba las clases `dataloader` y `cleaner` del script `limpieza.py` utilizando un archivo Excel ubicado en `data/raw/`.

## 0. Configuración de rutas

In [ ]:
import sys
from pathlib import Path

# Agregar la carpeta 'src' al path para importar limpieza.py
ruta_src_datacleaning = Path('..') / 'src' / 'datacleaning'
sys.path.append(str(ruta_src_datacleaning))

# Ruta al archivo Excel de prueba
RUTA_ARCHIVO = Path('..') / 'data' / 'raw' / 'copia_calidad_docentes.xlsx'  # <- Ajustar al nombre real del archivo
COLUMNA_CLAVE = 'Si desea complementar sus respuestas o hacer sugerencias para nuestro mejoramiento continuo, por favor inclúyalas a continuación (si no tiene comentarios, dejar este espacio en blanco):'                                      # <- Ajustar al nombre real de la columna
HOJA = 0                                                     # <- Índice o nombre de la hoja Excel
RUTA_SALIDA = Path('..') / 'data' / 'processed' / 'datos_limpios_prueba1.csv'

print(f"Ruta archivo : {RUTA_ARCHIVO.resolve()}")
print(f"Archivo existe: {RUTA_ARCHIVO.exists()}")

## 1. Importar módulo

In [ ]:
# importar las clases desde el .py
from datacleaning.limpieza import Cleaner

print("Módulo importado correctamente.")

## 2. Prueba de `dataloader` — carga del archivo Excel

## 2.1. Prueba carga - sin recurrir a la clase

> Verificar cómo reconoce Pandas la columna clave

In [ ]:
import pandas as pd

df_prueba1 = pd.read_excel(RUTA_ARCHIVO, sheet_name=HOJA)
df_prueba1.info()  # Mostrar información del DataFrame antes de la limpieza

In [ ]:
df_prueba1.columns

In [ ]:
print(f"DEBUG: El dtype es: {df_prueba1[COLUMNA_CLAVE].dtype}")
print(f"DEBUG: Los primeros 5 valores son: {df_prueba1[COLUMNA_CLAVE].head().tolist()}")

## 2.2. Prueba de carga con la clase `Cleaner`

In [ ]:
cleaner = Cleaner(
    file_path=RUTA_ARCHIVO,
    survey_name="stopwords_calidaddocentes", #Calidad Docentes
    key_column=COLUMNA_CLAVE,
    sheet_name=HOJA
)

df = cleaner.load_data()

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"Columnas disponibles: {df.columns.tolist()}")
df.head()

## 3. Prueba de `cleaner.limpiar_texto` — limpieza unitaria

In [ ]:
# Casos de prueba individuales
casos = [
    "  ¡Hola Mundo!  ",
    "Ángela estudió en la Universidad Académica",
    "Texto con: caracteres@especiales #raros!",
    "España y la ñ deben conservarse",
]

print(f"{'Texto original':<45} -> {'Texto limpio'}")
print("-" * 75)
for caso in casos:
    print(f"{repr(caso):<45} -> {repr(cleaner._clean_text(caso))}")

## 4. Prueba de `cleaner.limpiar_columna_clave` — limpieza sobre el DataFrame

In [ ]:
df_limpio = cleaner.clean_key_column()

print(f"Columna nueva generada: '{COLUMNA_CLAVE}_clean'")
df_limpio[[COLUMNA_CLAVE, f"{COLUMNA_CLAVE}_clean"]].head(10)

## 5. Prueba de `cleaner.save_cleaned_data` — exportar a CSV

In [ ]:
# Crear carpeta de salida si no existe
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

cleaner.save_cleaned_data(out_path=str(RUTA_SALIDA))

print(f"Archivo guardado en: {RUTA_SALIDA.resolve()}")

In [ ]:
import pandas as pd
datos_limpios = pd.read_csv(RUTA_SALIDA)

In [ ]:

datos_limpios.dropna(how='all').iloc[:, -2:].sort_values(by=datos_limpios.columns[-1])  # Mostrar las dos últimas columnas

## 6. Prueba de manejo de errores

In [ ]:
# 6.1 Archivo inexistente
print("--- Archivo inexistente ---")
try:
    loader_err = Cleaner(file_path='no_existe.xlsx', key_column=COLUMNA_CLAVE,
                         survey_name="stopwords_calidaddocentes", sheet_name=HOJA)
    loader_err.load_data()
except (FileNotFoundError, ValueError) as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 6.2 Columna clave inexistente
print("\n--- Columna clave inexistente ---")
try:
    loader_err = Cleaner(file_path=RUTA_ARCHIVO, key_column='columna_falsa',
                    survey_name="stopwords_calidaddocentes", sheet_name=HOJA)

    loader_err.load_data()
except ValueError as e:
    print(f"[OK] Error capturado: {e}")

In [ ]:
# 6.3 Formato no soportado
print("\n--- Formato no soportado ---")
try:
    loader_err = Cleaner(file_path='archivo.txt', key_column=COLUMNA_CLAVE,
                    survey_name="stopwords_calidaddocentes", sheet_name=HOJA)

    loader_err.load_data()
except (FileNotFoundError, ValueError) as e:
    print(f"[OK] Error capturado: {e}")